In [ ]:
import pandas as pd
import numpy as np
import arviz as az
from jax import numpy as jnp
import matplotlib.pyplot as plt

from summer3.epi import CategoryData, strat_data_from_pandas

from tb_macro.constants import (
    ALL_COMPARTMENTS,
    AGE_STRATA,
    MAX_AGE,
    DATA_PATH,
    ISO3,
    START_TIME,
    END_TIME,
    YOUNG_END_AGE,
    SOLVER_KWARGS,
)
from tb_macro.epi import get_base_model, add_natural_history, add_seeding, add_latency_flows, add_infection_flows
from tb_macro.inputs import (
    get_country_pop,
    get_single_age_pop_from_ungroups,
    get_group_popsizes,
    get_un_mortality,
    add_groups_to_single_pop,
    build_age_weight_lookup,
    get_fertility_data,
    calc_tsr_from_outcomes,
    calc_death_in_unsucc_outcomes,
    get_country_indicators,
)
from tb_macro.demography import add_replacement_deaths, add_ageing_flows, prepare_pop_data_for_entries, add_entry_births
from tb_macro.health_system import add_treatment_flows, add_detection
from tb_macro.parameters import BASE_PARAMS
from tb_macro.targets import NOTIF_TARGET, LATENT_TARGET
from tb_macro.outputs import (
    get_share_folder_file_path,
    get_age_inc,
    get_age_prev,
    get_age_latent,
    get_age_notifs,
    get_age_deaths,
    get_total_pop,
)
from tb_macro.plotting import plot_outputs

plt.style.use("ggplot")
pd.options.plotting.backend = "matplotlib"

In [ ]:
# Data loading and processing
pop_data = get_country_pop(ISO3)
single_age_pops = get_single_age_pop_from_ungroups(pop_data)
group_popsize = get_group_popsizes(single_age_pops)
mort_data = get_un_mortality(ISO3)
death_rates = mort_data.div(group_popsize, axis=0).dropna()
add_groups_to_single_pop(single_age_pops)
age_weights = build_age_weight_lookup(single_age_pops)
fert = get_fertility_data(ISO3)
fert_padded = fert.reindex(columns=range(MAX_AGE + 1), fill_value=0.0)
raw_outcome_data = pd.read_csv(DATA_PATH / "who/who_outcomes_20260514T0437Z.csv")
outcome_data = raw_outcome_data[raw_outcome_data["iso3"] == ISO3]
tsr = calc_tsr_from_outcomes(outcome_data)
death_in_unsucc = calc_death_in_unsucc_outcomes(outcome_data)
who_indicators = get_country_indicators(ISO3)
who_mort = who_indicators["e_mort_tbhiv_num"] + who_indicators["e_mort_exc_tbhiv_num"]

In [ ]:
# Model construction
epi_model, disease_state, age_strat, clin_strat, infect_strat = get_base_model(START_TIME, END_TIME)
start_apops = [1000.0] * len(AGE_STRATA) # Arbitrary starting values, inflows determine growth
entry_times, entry_rates = prepare_pop_data_for_entries(group_popsize, START_TIME, sum(start_apops))

add_infection_flows(epi_model, disease_state, age_strat, clin_strat, infect_strat, age_weights, group_popsize, fert_padded, YOUNG_END_AGE, START_TIME)
add_natural_history(epi_model, disease_state, age_strat, clin_strat, infect_strat)
add_ageing_flows(epi_model, age_strat)
add_seeding(epi_model, disease_state, START_TIME)
add_detection(epi_model, disease_state, clin_strat, START_TIME)
add_replacement_deaths(epi_model, disease_state, age_strat, death_rates, START_TIME)
add_entry_births(epi_model, disease_state, age_strat, START_TIME, entry_rates, entry_times)
add_treatment_flows(death_rates, START_TIME, epi_model, disease_state, age_strat, infect_strat, clin_strat, tsr, death_in_unsucc)
add_latency_flows(epi_model, disease_state, age_strat, clin_strat, infect_strat)

# Initialisation
init_apops_series = pd.Series(index=[str(a) for a in AGE_STRATA], data=np.array(start_apops))
init_apops = strat_data_from_pandas(init_apops_series, age_strat)
init_dpops = [0.0] * len(ALL_COMPARTMENTS)
init_dpops[ALL_COMPARTMENTS.index("mtb_naive")] = 1.0
pop_splits = [CategoryData(disease_state.categories(), jnp.array((init_dpops)))]
epi_model.set_initial_population(init_apops, pop_splits)
epi_model.computed_values.append("dynamic_mm")

In [ ]:
idata = az.from_netcdf("nuts_100_100_idata.nc")
az.summary(idata)

In [ ]:
# Load an idata that was prepared earlier
idata = az.from_netcdf("nuts_100_100_idata.nc")

In [ ]:
n_samples = 5
posterior = idata.posterior.stack(sample=("chain", "draw"))
idxs = np.random.choice(posterior.sizes["sample"], size=n_samples, replace=False)
samples = posterior.isel(sample=idxs)

In [ ]:
# Collate outputs
scen_params = [{}, {"detect_gap_reduction": 0.5}]
sample_labels = []
output_funcs = {
    "incidence": get_age_inc,
    "prevalence": get_age_prev,
    "latent": get_age_latent,
    "notifications": get_age_notifs,
    "deaths": get_age_deaths,
    "total_pop": get_total_pop,
}
outputs = [{out: [] for out in output_funcs} for _ in scen_params]

for i in range(len(idxs)):
    run = f"chain_{int(samples['chain'][i])}/draw_{int(samples['draw'][i])}"
    sample_labels.append(run)
    c_params = {k: float(samples[k].isel(sample=i)) for k in idata.posterior.data_vars}
    for s, s_params in enumerate(scen_params):
        results = epi_model.run(BASE_PARAMS | c_params | s_params, solver_kwargs=SOLVER_KWARGS)
        for out, func in output_funcs.items():
            outputs[s][out].append(func(results, age_strat, disease_state).to_pandas_df())

# Collate into single dataframe
full_outs = []
for s in range(len(scen_params)):
    scen_outs = []
    for out in output_funcs:
        scen_outs.append(pd.concat(outputs[s][out], axis=1, keys=sample_labels))
    full_outs.append(pd.concat(scen_outs, axis=1, keys=output_funcs.keys()))
full_out = pd.concat(full_outs, axis=1, keys=range(len(scen_params)))

In [ ]:
def sum_df_over_lower_level(df):
    return df.T.groupby(level=0).sum().T

s_plot = 0
total_pop = sum_df_over_lower_level(full_out[s_plot]["total_pop"])
incs = sum_df_over_lower_level(full_out[s_plot]["incidence"])
notifs = sum_df_over_lower_level(full_out[s_plot]["notifications"])
prevs = sum_df_over_lower_level(full_out[s_plot]["prevalence"])
tb_deaths = sum_df_over_lower_level(full_out[s_plot]["deaths"])
latent = sum_df_over_lower_level(full_out[s_plot]["latent"])

In [ ]:
fig = plot_outputs(prevs, incs, notifs, NOTIF_TARGET, tb_deaths, who_mort, latent, None, total_pop, 1980.0, 2050.0, "count")

In [ ]:
fig = plot_outputs(prevs, incs, notifs, NOTIF_TARGET, tb_deaths, None, latent, LATENT_TARGET, total_pop, 1980.0, 2050.0, "rate")

In [ ]:
# james_work_gdrive = "/Users/jtrauer/Library/CloudStorage/GoogleDrive-james.trauer@monash.edu/"
# out_path = get_share_folder_file_path(james_work_gdrive) 
# full_out.to_csv(out_path / "full_outputs.csv")